In [1]:
from tqdm.notebook import tqdm as tqm
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from utils.func_aux import *
import plotly.express as px
import scipy.stats as st
import seaborn as sns
import pandas as pd
import numpy as np
import shutil
import time 
import os

# Opciones de matplotlib
rc = plt.rcParams
rc["figure.figsize"] = [15, 5]

# Para mostrar todas las columnas cuando se imprime un df
pd.set_option("display.max_columns", None)

# Para poner el estilo de las gráficas de matplotlib parecido al de ggplot
plt.style.use("ggplot") 

go_to_PCUI_Proyect()

# Cálculo de Indicadores


Tenemos tres maneras de obtener diferentes indicadores:
- PCUI 
- Assessment
- pymoo

## Calculando Indicadores con PCUI-Project


Se optó por hacerlo con el proyecto de [Assessment](#calculando-con-assessment), pero se deja esta parte para referencia

1. Se hace la simulación con los pesos correspondientes y se obtiene el archivo de poblaciones `.pof`
2. Se obtienen las soluciones no dominadas mediante `./emo_ndset output/NSGA2_ZDT1_02D_R01.pof`
3. Se calculan los indicadores mediante la instrucción ` ./emo_indicator HV output/NSGA2_ZDT1_02D_R01.pof.nd 1 1.2 1.2` (cada uno de los indicadores puede requerir argumentos de entrada, como el punto a partir del cuál calcular en el Hipervolumen)


In [7]:
def get_QI_terminal_PCUI(
    indicator: str, problema: str, w0: int, n_objetivos: int, run: int,*args, nd: bool = False
):
    """Para calcular el indicador de un archivo usando el emo_indicator de PCUI
    los args se refieren a los puntos de referencia que a veces necesitan los QI
    """
    go_to_PCUI_Proyect()
    # Ponerle el archivo generado en el path correcto
    path_sol = f"../../archivos_w/w_0{w0}/PCUIEMOA_{problema.upper()}_{str(n_objetivos).zfill(2)}D_R{str(run).zfill(2)}.pof"

        
    if nd:
        path_sol += ".nd"
    cadena_ej = f"./demo/emo_indicator {indicator.upper()} {path_sol}"
    for arg in args:
        cadena_ej+=f' {arg}'

    return cadena_ej


indicador, problema, w0, n_objetivos, run = "s-energy", "DTLZ1", 5, 5, 1
comando=get_QI_terminal_PCUI(indicador, problema, w0, n_objetivos, run,1,nd=False)
print(comando)
os.system(comando)

./demo/emo_indicator S-ENERGY ../../archivos_w/w_05/PCUIEMOA_DTLZ1_05D_R01.pof 1
tmp ../../archivos_w/w_05/PCUIEMOA_DTLZ1_05D_R01.pof, nrun 1
1 ../../archivos_w/w_05/PCUIEMOA_DTLZ1_05D_R01.pof 4.501831e+06


0

# Calculando con Assessment


Primero se mueven los archivos desde donde fueron generados (PCUI output) a una carpeta donde se puedan leer individualmente. Esa carpeta se pasa a path_archivos para poder calcular los indicadores

1. Se configuran los parámetros del  
   Se corre el programa `evaluate.sh` sobre el directorio donde están las poblaciones `.pof`

2. Se obtienen los archivos con terminación .QI por cada uno de los indicadores


In [3]:
def get_QI_assesment(MOP_lista: list, dim_lista: list, ind_lista: list, path_archivos: str, runs:int=10):
    """Sobrescribe el archivo de evaluate.sh para evaluar la lista de problemas. El path archivos es desde donde se encuentra el evaluate.sh
    Se le pueden poner sólo los indicadores que se quieran, pero eso para el futuro.
    path_archivos: la ruta relativa desde excecute.sh a la lista de archivos .pof a correr
    Al correr el programa
    Genera los archivos en el mismo directorio con terminación .hv .igd .igd+ .eps+ .r2 .s-energy .spd
    """

    go_to_Assesment()
    dicc_str_indicadores = {
        "HV": "\t\t\t./emo_indicator HV $data $runs $refPoint\n",
        "S-ENERGY": "\t\t\t./emo_indicator S-ENERGY $data $runs $dim\n",
        "SPD": "\t\t\t./emo_indicator SPD $data $runs 10 \n",
        "IGD": "\t\t\t./emo_indicator IGD $data $runs 2 $refset\n",
        "IGD+": "\t\t\t./emo_indicator IGD+ $data $runs $refset\n",
        "EPS+": "\t\t\t./emo_indicator EPS+ $data $runs $refset\n",
        "DELTAP": "\t\t\t./emo_indicator DELTAP $data $runs 2 $refset\n",
        "R2": "\t\t\t./emo_indicator R2 $data $runs $wfile vector_angle_distance_scaling\n",
        "PD": "\t\t\t./emo_indicator PD $data $runs\n",
    }
    cadena_indicadores = "".join(
        [dicc_str_indicadores[ind.upper()] for ind in ind_lista]
    )
    cadena_sh = f"""#! /bin/bash
# Esta parte es sólo para los escalables

# Genera un archivo en el directorio output con el nombre del algoritmo extensión .indicador
# en ese archivo se encuentran las mediciones de cada una de las corridas y el indicador

MOP=({" ".join(MOP_lista)})
DIM=({" ".join([str(i) for i in dim_lista])})

ALG=(PCUIEMOA)
runs={runs}
N=120
"""
    cadena_sh += """
for alg in ${ALG[*]}
do
	for dim in ${DIM[*]}
	do
		for pom in ${MOP[*]}
		do
	"""

    cadena_sh += f"""
			printf -v data "{path_archivos}/%s_%s_%.2dD" "$alg" "$pom" "$dim"
			refPoint=""

			if test $pom == DTLZ2 -o $pom == DTLZ3 -o $pom == DTLZ4 -o $pom == DTLZ5 -o $pom == DTLZ6
			then
				for (( i=1; i<= $dim; i++ ))
				do
					refPoint+=" 2"		
				done
			elif test $pom == DTLZ1_MINUS -o $pom == DTLZ2_MINUS -o $pom == DTLZ3_MINUS -o $pom == DTLZ4_MINUS -o $pom == DTLZ5_MINUS -o $pom == DTLZ6_MINUS -o $pom == WFG1_MINUS -o $pom == WFG2_MINUS -o $pom == WFG3_MINUS -o $pom == WFG4_MINUS -o $pom == WFG5_MINUS -o $pom == WFG6_MINUS -o $pom == WFG7_MINUS -o $pom == WFG8_MINUS -o $pom == WFG9_MINUS
			then
				for (( i=1; i<= $dim; i++ ))
				do
					refPoint+=" 1"		
				done
			elif test $pom == DTLZ7_MINUS
			then
				for (( i=1; i<$dim; i++ ))
				do
					refPoint+=" 0.1"		
				done	
				refPoint+=" -10 "	
			elif test $pom == DTLZ7
			then
				for (( i=1; i<$dim; i++ ))
				do
					refPoint+=" 1"		
				done	
				refPoint+=" 21"	
			elif test $pom == WFG1 -o $pom == WFG2 -o  $pom == WFG3 -o  $pom == WFG4 -o  $pom == WFG5 -o  $pom == WFG6 -o  $pom == WFG7 -o  $pom == WFG8 -o  $pom == WFG9
			then
				for (( i=1; i<= $dim; i++ ))
				do
					aux=$((2*$i+1)) 
					refPoint+=" "
					refPoint+=$aux		
				done
			elif test $pom == DTLZ1
			then
				for (( i=1; i <= $dim; i++ ))
				do
					refPoint+=" 1"
				done
			fi
			
			printf -v refset "refsets/%s_%.2dD.pof" "$pom" "$dim"
			printf -v wfile "input/weight/weight_%.2dD_%d.udh" "$dim" "$N"
"""

    cadena_sh += cadena_indicadores
    cadena_sh += """
		done
	done
done	
	"""
    
    with open(file="./demo/evaluate.sh", mode="w") as f:
        f.write(cadena_sh)
	
    lista_salidas = []

    for prob in MOP_lista:
        for dim in dim_lista:
            for ind in ind_lista:
                lista_salidas.append(
                    f"PCUIEMOA_{prob}_{str(dim).zfill(2)}D.{ind.lower()}"
                )
        
    
    os.chdir('./demo')
    os.system('./evaluate.sh')
    
    go_to_Assesment()

    return [f'{path_archivos[1:]}/{li}' for li in lista_salidas]


In [4]:
MOP_lista=['DTLZ1', 'DTLZ2', 'DTLZ3', 'DTLZ4', 'DTLZ5', 'DTLZ6', 'DTLZ7',
       'WFG1', 'WFG2', 'WFG3', 'WFG4', 'WFG5', 'WFG6', 'WFG7', 'WFG8',
       'WFG9']

dim_lista=[2,3,4,5,6,7]
lista_indicadores = ['hv','eps+','igd','igd+','r2','s-energy','spd']

for i in tqm(list(range(11))):
	paths_indicadores = get_QI_assesment(
	MOP_lista=MOP_lista,
		dim_lista=dim_lista,
		ind_lista=lista_indicadores,
		path_archivos=f"../../archivos_w_IGDp_20_runs/w_0{i}",
		runs=20
	)


# dim_lista=[2]
# MOP_lista=['ZDT1', 'ZDT2', 'ZDT3', 'ZDT4', 'ZDT6']

# for i in tqm(range(11)):
# 	paths_indicadores = get_QI_assesment(
# 	MOP_lista=MOP_lista,
# 		dim_lista=dim_lista,
# 		ind_lista=lista_indicadores,
# 		path_archivos=f"../../archivos_w_R2/w_0{i}"
# 	)


  0%|          | 0/11 [00:00<?, ?it/s]

In [5]:

paths_indicadores

['./../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.hv',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.eps+',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.igd',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.igd+',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.r2',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.s-energy',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_02D.spd',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.hv',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.eps+',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.igd',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.igd+',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.r2',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.s-energy',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_03D.spd',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_04D.hv',
 './../archivos_w_IGDp_20_runs/w_010/PCUIEMOA_DTLZ1_04D.eps+'

# Poniendo todos los indicadores en una misma tabla

In [4]:
go_to_Assesment()

# lista_indicadores = ['hv','eps+','igd','igd+','r2','s-energy','spd']
# problema_lista, ind_lista, wi_lista, dim_lista,run_lista,value_lista=[],[],[],[],[],[]

# ws=[f'w_0{i}' for i in range(11)]
# for wsi in ws:
#     for file in os.listdir(f'../archivos_w_R2/{wsi}'):
#         ind=file.split('.')[-1]
#         if  ind in lista_indicadores:
#             _,prob, dim_ind= file.split('_')
#             dim=int(dim_ind[:2])
#             problema_lista+=[prob]*10
#             ind_lista+=[ind]*10
#             wi_lista+=[wsi]*10
#             dim_lista+=[dim]*10
            
#             run_lista+=[i for i in range(10)]
#             value_lista+=list(pd.read_csv(f'../archivos_w_R2/{wsi}/{file}').iloc[:,0].values)


In [5]:
# df_todos_indicadores=pd.DataFrame({'problema':problema_lista, 'indicador':ind_lista, 'w0':wi_lista, 'n_objetivos':dim_lista,'run':run_lista,'valor':value_lista})
# df_todos_indicadores.to_csv('../tablas_generadas/todos_QI_R2_senergy.csv',index=False)
# df_todos_indicadores_1=pd.read_csv('../tablas_generadas/todos_QI_IGDp_senergy.csv')

# df_todos_indicadores.loc[:,'hiperparam_conv']='R2'
#! para concatenar las dos tablas pd.concat(df_todos_indicadores.df_todos_indicadores_1).to_csv('../tablas_generadas/todos_QI.csv,index=False)

Estos número no coinciden porque no se está calculando el HV de los ZDT porque no se tiene un conjunto de referencia. Para IGD e IG+ llama nan, pero HV no se ejecuta

In [9]:
go_to_Assesment()
df_todos_indicadores=pd.read_csv('df_todos_indicadores_1')
df_todos_indicadores.rename(columns={'dimension': 'n_objetivos','valor':'valor_indicador'}).to_csv('../tablas_generadas/todos_QI_R2_senergy.csv',index=False)

In [15]:
df_todos_indicadores=pd.read_csv('../tablas_generadas/todos_QI_v2.csv')
# df_todos_indicadores.rename(columns={'dimension': 'n_objetivos','valor':'valor_indicador'})

In [13]:
pd.read_csv('../tablas_generadas/todos_QI.csv')

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
1,IGD+,WFG9,3,0.001,1,eps+,0.240986
2,IGD+,WFG9,3,0.001,2,eps+,0.240616
3,IGD+,WFG9,3,0.001,3,eps+,0.236098
4,IGD+,WFG9,3,0.001,4,eps+,0.267390
...,...,...,...,...,...,...,...
147835,R2,WFG9,6,0.999,5,hv,93347.990000
147836,R2,WFG9,6,0.999,6,hv,97336.020000
147837,R2,WFG9,6,0.999,7,hv,95928.310000
147838,R2,WFG9,6,0.999,8,hv,98113.420000


Valores nulos

In [7]:
pd.set_option('display.max_rows', None)
df_todos_indicadores[df_todos_indicadores.valor.astype(float).isna()].groupby(['problema','indicador','n_objetivos']).size().reset_index()

,problema,indicador,dimension,0


In [8]:
# indicadores que son 0 todos
df_todos_indicadores[df_todos_indicadores.valor.astype(float)<0.005].groupby(['problema','indicador','n_objetivos']).size().reset_index()

,problema,indicador,dimension,0
0,DTLZ1,eps+,2,110
1,DTLZ1,igd,2,110
2,DTLZ1,igd+,2,110
3,DTLZ2,igd,2,110
4,DTLZ2,igd+,2,110
5,DTLZ3,eps+,6,1
6,DTLZ3,eps+,7,1
7,DTLZ3,hv,7,19
8,DTLZ3,igd,2,48
9,DTLZ3,igd+,2,73


Valor muy pequeño de los de convergencia querría decir que el conjunto de referencia es casi idéntico al otro. 